# Teaching transformers how to write like Shakespeare

### Imports

In [2]:
import os
import sys
import numpy as np
import torch
from tqdm import tqdm
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve() / "src"))
from transformers.word_embedding import word_embedding

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Import dataset

In [3]:
input_file_path = "/Users/juanfrancisco/Desktop/Transformers/data/tinyshakespeare/"

# # download the tiny shakespeare dataset
# if not os.path.exists(input_file_path + "input.txt"):
#     data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
#     with open(input_file_path + "input.txt", 'w', encoding='utf-8') as f:
#         f.write(requests.get(data_url).text)

with open(input_file_path + "input.txt", 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:1000])
print(f"length of dataset in characters: {len(text)}")

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



### Word embedding

In [4]:
text = text[:100]

model = word_embedding(embedding_type="gpt2")
tokens = model.encode(text)
embedding = model.embed(text)
data = torch.tensor(tokens, dtype=torch.long)

# Split training and validation dataset
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(f"train has {len(train_data):,} tokens")
print(f"val has {len(val_data):,} tokens")

train has 27 tokens
val has 4 tokens


Export embeddings

In [5]:
# export to bin files
train_ids = np.array(train_data, dtype=np.uint16)
val_ids = np.array(val_data, dtype=np.uint16)
train_ids.tofile(os.path.join(input_file_path, 'train.bin'))
val_ids.tofile(os.path.join(input_file_path, 'val.bin'))

/var/folders/cr/h67rdg2s2097w50n4kgbw6z00000gn/T/ipykernel_29789/2280340314.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  train_ids = np.array(train_data, dtype=np.uint16)
/var/folders/cr/h67rdg2s2097w50n4kgbw6z00000gn/T/ipykernel_29789/2280340314.py:3: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  val_ids = np.array(val_data, dtype=np.uint16)


### Training

In [6]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([5962]) the target: 22307
when input is tensor([ 5962, 22307]) the target: 25
when input is tensor([ 5962, 22307,    25]) the target: 198
when input is tensor([ 5962, 22307,    25,   198]) the target: 8421
when input is tensor([ 5962, 22307,    25,   198,  8421]) the target: 356
when input is tensor([ 5962, 22307,    25,   198,  8421,   356]) the target: 5120
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120]) the target: 597
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597]) the target: 2252


In [7]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[ 502, 2740,   13,  198,  198, 3237,   25,  198],
        [  25,  198, 5248,  461,   11, 2740,   13,  198],
        [  11, 3285,  502, 2740,   13,  198,  198, 3237],
        [ 502, 2740,   13,  198,  198, 3237,   25,  198]])
targets:
torch.Size([4, 8])
tensor([[2740,   13,  198,  198, 3237,   25,  198, 5248],
        [ 198, 5248,  461,   11, 2740,   13,  198,  198],
        [3285,  502, 2740,   13,  198,  198, 3237,   25],
        [2740,   13,  198,  198, 3237,   25,  198, 5248]])


In [ ]:
from torch.nn import functional as F

class Head(torch.nn.Module):
    """ one head of self attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = torch.nn.Linear(vocab_dim, head_size, bias=False)
        self.query = torch.nn.Linear(vocab_dim, head_size, bias=False)
        self.value = torch.nn.Linear(vocab_dim, head_size, bias=False)

    def forward(self, x):

        B, T, C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)
        v = self.value(x) # (B,T,head_size)

        weight = torch.matmul(q, k.transpose(-2, -1)) / k.shape[-1]**0.5
        mask = torch.triu(torch.ones(head_size, head_size, device=weight.device, dtype=torch.bool), diagonal=1)
        weight = weight.masked_fill(mask, float("-inf"))
        weight = torch.softmax(weight, dim=-1)

        out = weight @ v

        return out


class GPTLanguageModel(torch.nn.Module):
    def __init__(self, vocab_size, vocab_dim):
        super().__init__()
        self.token_embedding_table = torch.nn.Embedding(vocab_size, vocab_dim) # holds semantic value
        self.position_embedding_table = torch.nn.Embedding(1000, vocab_dim) 
        self.lm_head = torch.nn.Linear(vocab_dim, vocab_size)
        self.sa_head = Head(vocab_dim)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embeddings = self.token_embedding_table(idx) # (B,T,C)
        position_embeddings = self.position_embedding_table(torch.arange(T)) # (T,C)
        x = token_embeddings + position_embeddings # (B,T,C)
        x = self.sa_head(x)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = torch.nn.functional.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
    
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [76]:
vocab_size = embedding.num_embeddings
vocab_dim = 32
epochs = 10

# create a PyTorch optimizer
prediction_model = GPTLanguageModel(vocab_size, vocab_dim)
optimizer = torch.optim.AdamW(prediction_model.parameters(), lr=1e-3)

for steps in tqdm(range(epochs)): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = prediction_model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)

    #update parameters
    loss.backward()
    optimizer.step()

    print(f"Iteration {steps}: loss = {loss.item()}")

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 171.14it/s]

Iteration 0: loss = 10.897692680358887
Iteration 1: loss = 10.945741653442383
Iteration 2: loss = 10.867016792297363
Iteration 3: loss = 10.834026336669922
Iteration 4: loss = 10.794642448425293
Iteration 5: loss = 10.732468605041504
Iteration 6: loss = 10.717191696166992
Iteration 7: loss = 10.715497016906738
Iteration 8: loss = 10.659035682678223
Iteration 9: loss = 10.626810073852539
